In [3]:
"""
generate_cdr_data.py
Generates synthetic CDR (Consumer Data Right) open banking data.
Produces 5 CSV files simulating realistic data holder feeds.
"""

import random
import csv
from datetime import datetime, timedelta
from faker import Faker

fake = Faker('en_AU')
random.seed(42)
Faker.seed(42)

OUTPUT_DIR = "data/raw"
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── CONFIG ──────────────────────────────────────────────────────────────
NUM_CUSTOMERS   = 200
NUM_ACCOUNTS    = 320
NUM_CONSENTS    = 280
NUM_TRANSACTIONS = 5000
NUM_FEED_LOGS   = 400

DATA_HOLDERS = [
    "Commonwealth Bank", "Westpac", "ANZ", "NAB",
    "Bendigo Bank", "ING Australia", "Macquarie Bank"
]

ACCOUNT_TYPES = ["savings", "transaction", "term_deposit", "mortgage", "credit_card"]
CONSENT_STATUSES = ["active", "active", "active", "revoked", "expired"]  # weighted
TRANSACTION_TYPES = ["debit", "credit"]
FEED_STATUSES = ["success", "success", "success", "failed", "partial"]   # weighted

# ── 1. CUSTOMERS ─────────────────────────────────────────────────────────
print("Generating customers...")
customers = []
customer_ids = [f"CUST_{str(i).zfill(5)}" for i in range(1, NUM_CUSTOMERS + 1)]

for cid in customer_ids:
    customers.append({
        "customer_id": cid,
        "full_name": fake.name(),
        "email": fake.email(),
        "phone": fake.phone_number(),
        "state": random.choice(["NSW", "VIC", "QLD", "WA", "SA", "ACT"]),
        "created_at": fake.date_time_between(start_date="-3y", end_date="-6m").isoformat(),
    })

with open(f"{OUTPUT_DIR}/customers.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=customers[0].keys())
    writer.writeheader()
    writer.writerows(customers)

# ── 2. ACCOUNTS ──────────────────────────────────────────────────────────
print("Generating accounts...")
accounts = []
account_ids = [f"ACC_{str(i).zfill(6)}" for i in range(1, NUM_ACCOUNTS + 1)]

for aid in account_ids:
    owner = random.choice(customer_ids)
    holder = random.choice(DATA_HOLDERS)
    acc_type = random.choice(ACCOUNT_TYPES)
    balance = round(random.uniform(-500, 85000), 2)
    # Inject ~5% dirty data: null balance or future open_date
    if random.random() < 0.05:
        balance = None
    open_date = fake.date_between(start_date="-5y", end_date="today")
    if random.random() < 0.03:
        open_date = fake.date_between(start_date="today", end_date="+1y")  # future date anomaly

    accounts.append({
        "account_id": aid,
        "customer_id": owner,
        "data_holder": holder,
        "account_type": acc_type,
        "balance_aud": balance,
        "currency": "AUD",
        "status": random.choice(["open", "open", "open", "closed", "suspended"]),
        "open_date": open_date.isoformat(),
        "last_updated": fake.date_time_between(start_date="-30d", end_date="now").isoformat(),
    })

with open(f"{OUTPUT_DIR}/accounts.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=accounts[0].keys())
    writer.writeheader()
    writer.writerows(accounts)

# ── 3. CONSENTS ───────────────────────────────────────────────────────────
print("Generating consents...")
consents = []
consent_ids = [f"CONS_{str(i).zfill(6)}" for i in range(1, NUM_CONSENTS + 1)]

SCOPES = [
    "bank:accounts.basic:read",
    "bank:accounts.detail:read",
    "bank:transactions:read",
    "bank:payees:read",
    "common:customer.basic:read",
]

for cid in consent_ids:
    customer = random.choice(customer_ids)
    holder = random.choice(DATA_HOLDERS)
    status = random.choice(CONSENT_STATUSES)
    created = fake.date_time_between(start_date="-2y", end_date="-1d")
    expiry = created + timedelta(days=random.choice([90, 180, 365]))
    scopes = random.sample(SCOPES, k=random.randint(1, 4))

    # Inject ~4% dirty: missing customer_id
    cust_val = customer if random.random() > 0.04 else None

    consents.append({
        "consent_id": cid,
        "customer_id": cust_val,
        "data_holder": holder,
        "status": status,
        "scopes": "|".join(scopes),
        "created_at": created.isoformat(),
        "expires_at": expiry.isoformat(),
        "revoked_at": fake.date_time_between(start_date=created, end_date="now").isoformat() if status == "revoked" else None,
        "sharing_duration_days": (expiry - created).days,
    })

with open(f"{OUTPUT_DIR}/consents.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=consents[0].keys())
    writer.writeheader()
    writer.writerows(consents)

# ── 4. TRANSACTIONS ───────────────────────────────────────────────────────
print("Generating transactions...")
transactions = []
open_accounts = [a["account_id"] for a in accounts if a["status"] == "open"]

MERCHANTS = [
    "Woolworths", "Coles", "Bunnings", "JB Hi-Fi", "Kmart",
    "Shell", "BP", "Uber Eats", "Netflix", "Spotify",
    "AGL Energy", "Sydney Water", "Ausgrid", "Rent Payment", "Salary"
]

for i in range(1, NUM_TRANSACTIONS + 1):
    acc = random.choice(open_accounts)
    txn_type = random.choice(TRANSACTION_TYPES)
    amount = round(random.uniform(1.5, 4500), 2)
    # Inject ~3% duplicates (same txn_id different row — data quality issue)
    txn_id = f"TXN_{str(random.randint(1, NUM_TRANSACTIONS - 50)).zfill(7)}" if random.random() < 0.03 else f"TXN_{str(i).zfill(7)}"

    transactions.append({
        "transaction_id": txn_id,
        "account_id": acc,
        "transaction_type": txn_type,
        "amount_aud": amount,
        "currency": "AUD",
        "merchant_name": random.choice(MERCHANTS),
        "transaction_date": fake.date_time_between(start_date="-90d", end_date="now").isoformat(),
        "posted_date": fake.date_time_between(start_date="-90d", end_date="now").isoformat(),
        "description": fake.bs(),
        "status": random.choice(["settled", "settled", "settled", "pending", "failed"]),
    })

with open(f"{OUTPUT_DIR}/transactions.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=transactions[0].keys())
    writer.writeheader()
    writer.writerows(transactions)

# ── 5. DATA FEED LOGS ─────────────────────────────────────────────────────
print("Generating data feed logs...")
feed_logs = []

for i in range(1, NUM_FEED_LOGS + 1):
    holder = random.choice(DATA_HOLDERS)
    status = random.choice(FEED_STATUSES)
    run_time = fake.date_time_between(start_date="-60d", end_date="now")
    records_expected = random.randint(100, 2000)
    records_received = records_expected if status == "success" else random.randint(0, records_expected)
    latency_ms = random.randint(120, 8000) if status != "failed" else None
    error_code = random.choice(["TIMEOUT", "SCHEMA_MISMATCH", "AUTH_FAILURE", "RATE_LIMITED"]) if status == "failed" else None
    # Inject ~6% missing latency on partial feeds
    if status == "partial" and random.random() < 0.06:
        latency_ms = None

    feed_logs.append({
        "feed_log_id": f"FEED_{str(i).zfill(5)}",
        "data_holder": holder,
        "feed_type": random.choice(["accounts", "transactions", "consents"]),
        "run_at": run_time.isoformat(),
        "status": status,
        "records_expected": records_expected,
        "records_received": records_received,
        "latency_ms": latency_ms,
        "error_code": error_code,
        "ingestion_version": f"v{random.randint(1,3)}.{random.randint(0,9)}.{random.randint(0,9)}",
    })

with open(f"{OUTPUT_DIR}/feed_logs.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=feed_logs[0].keys())
    writer.writeheader()
    writer.writerows(feed_logs)

# ── SUMMARY ───────────────────────────────────────────────────────────────
print("\n✅ Synthetic CDR data generated successfully!")
print(f"   📁 Output folder: {OUTPUT_DIR}/")
print(f"   👤 customers.csv       — {len(customers):>5} rows")
print(f"   🏦 accounts.csv        — {len(accounts):>5} rows  (~5% dirty: null balance / future dates)")
print(f"   🔐 consents.csv        — {len(consents):>5} rows  (~4% dirty: missing customer_id)")
print(f"   💳 transactions.csv    — {len(transactions):>5} rows  (~3% duplicates)")
print(f"   📡 feed_logs.csv       — {len(feed_logs):>5} rows  (success / failed / partial feeds)")
print("\nData quality issues seeded intentionally for Step 3 validation checks.")

Generating customers...
Generating accounts...
Generating consents...
Generating transactions...
Generating data feed logs...

✅ Synthetic CDR data generated successfully!
   📁 Output folder: data/raw/
   👤 customers.csv       —   200 rows
   🏦 accounts.csv        —   320 rows  (~5% dirty: null balance / future dates)
   🔐 consents.csv        —   280 rows  (~4% dirty: missing customer_id)
   💳 transactions.csv    —  5000 rows  (~3% duplicates)
   📡 feed_logs.csv       —   400 rows  (success / failed / partial feeds)

Data quality issues seeded intentionally for Step 3 validation checks.
